# Домашнее задание: Трансформеры. Attention. Модели BERT, GPT
## Сравнение LSTM/GRU и BERT-классификации на задаче анализа тональности русскоязычных отзывов

**Данные:** Kaggle, «Kinopoisk's movies reviews» ([mikhailklemin/kinopoisks-movies-reviews](https://www.kaggle.com/datasets/mikhailklemin/kinopoisks-movies-reviews)).
Это **не один CSV**, а набор `.txt`-файлов вида `dataset/{neg,neu,pos}/<id фильма>-<номер отзыва>.txt`:
один файл — один отзыв, класс тональности — имя папки. Ноутбук читает данные как из распакованной папки,
так и **прямо из `archive.zip`** (распаковывать вручную не нужно).

**План ноутбука (по пунктам ДЗ):**
1. Загрузка и первичный разведочный анализ (EDA)
2. Предобработка данных (очистка, разбиение 70/15/15, веса классов, токенизация, словарь)
3. Базовая модель: двунаправленная 2-слойная LSTM/GRU (PyTorch)
4. BERT (HuggingFace): frozen encoder vs. fine-tuning
5. Сравнение моделей и выводы

## Как запустить в Google Colab
1. **Runtime → Change runtime type → T4 GPU** (нужен для BERT-части; на CPU всё тоже работает, но очень медленно —
   в этом случае уменьшите `SUBSET_TOTAL` в ячейке конфигурации).
2. **Данные** — подойдёт любой из способов, ноутбук сам найдёт архив:
   - загрузите `archive.zip` в корень «Моего диска» Google Drive — при запуске ячейки загрузки Colab запросит доступ к Drive;
   - *или* загрузите `archive.zip` в `/content` (панель **Files** слева → Upload; ~230 МБ, займёт несколько минут);
   - *или* ничего не загружайте — ноутбук попробует скачать датасет через `kagglehub`.
3. **Runtime → Run all** (один прогон сверху вниз).

*Ориентировочное время полного прогона на T4 — 10–15 минут (чтение 131 тыс. файлов ≈ 1 мин, обучение LSTM — секунды–минуты,
BERT frozen + fine-tuning — несколько минут).*

In [ ]:
# В Colab уже есть torch, sklearn, pandas, seaborn, matplotlib. Доустанавливаем только недостающее.
import importlib.util, subprocess, sys

for pkg in ['transformers', 'kagglehub']:
    if importlib.util.find_spec(pkg) is None:
        print('Устанавливаю', pkg)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Зависимости готовы.')

## 0. Импорты, воспроизводимость и конфигурация

Фиксируем seed для `random`, `numpy`, `torch`, выбираем устройство (GPU при наличии, иначе CPU)
и собираем все «ручки» эксперимента в одном месте.

In [ ]:
import os, re, sys, glob, time, copy, random, gc, warnings, zipfile, shutil
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

# ---------- воспроизводимость ----------
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed()

# ---------- устройство ----------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'GPU не найден')
if device.type == 'cpu':
    print('ВНИМАНИЕ: работаем на CPU. В Colab: Runtime -> Change runtime type -> T4 GPU. '
          'Иначе BERT будет обучаться очень долго (можно уменьшить SUBSET_TOTAL).')

IN_COLAB = 'google.colab' in sys.modules

# ---------- конфигурация эксперимента ----------
DATA_SOURCE = 'auto'                  # откуда брать данные, если их нет рядом с ноутбуком (только Colab):
                                      # 'auto' (Drive -> kagglehub -> upload) | 'drive' | 'kaggle' | 'upload'
KAGGLE_SLUG = 'mikhailklemin/kinopoisks-movies-reviews'
LABELS_ON_DISK = ['neg', 'neu', 'pos']  # имена папок с классами
MIN_CHARS = 10                        # отзывы короче удаляем (п. 2.1 ДЗ)
SUBSET_TOTAL = 5000                   # размер подвыборки для экспериментов (None = все данные; для BERT это ОЧЕНЬ долго)
BERT_MAX_LEN = 256                    # max_length для BERT (по ДЗ)
RNN_MAX_LEN = 256                     # максимальная длина последовательности (в словах) для LSTM/GRU

PALETTE = {'neg': '#d9534f', 'neu': '#f0ad4e', 'pos': '#5cb85c'}

# 1. Загрузка и первичный разведочный анализ (EDA)

Датасет лежит в виде `.txt`-файлов по папкам классов, поэтому «загрузка» состоит из трёх шагов:
1. **найти данные** — рядом с ноутбуком, в `/content`, в Google Drive (`/content/drive/MyDrive`) или в `/kaggle/input`;
   в Colab, если ничего не найдено, — смонтировать Drive → скачать через `kagglehub` → предложить загрузить архив вручную;
2. **прочитать все файлы** в один `pandas.DataFrame` (`text`, `label`, а также `movie_id` и `review_no` из имени файла);
3. **показать** `head()`, `columns`, `info()` и по паре примеров каждого класса — чтобы убедиться, что метки не перепутаны.

In [ ]:
def has_layout_dir(p):
    """В папке p есть подпапки neg / neu / pos?"""
    return os.path.isdir(p) and all(os.path.isdir(os.path.join(p, c)) for c in LABELS_ON_DISK)


def has_layout_zip(z):
    """В zip-архиве z лежат файлы в папках neg / neu / pos?"""
    try:
        with zipfile.ZipFile(z) as zf:
            parents = {n.split('/')[-2] for n in zf.namelist() if not n.endswith('/') and n.count('/') >= 1}
    except (zipfile.BadZipFile, OSError):
        return False
    return set(LABELS_ON_DISK) <= parents


def walk_for_layout(root, max_depth=4):
    """Рекурсивно (с ограничением глубины) ищем папку с neg/neu/pos."""
    root = os.path.abspath(root)
    base_depth = root.rstrip(os.sep).count(os.sep)
    for dirpath, dirnames, _ in os.walk(root):
        if has_layout_dir(dirpath):
            return dirpath
        if dirpath.rstrip(os.sep).count(os.sep) - base_depth >= max_depth:
            dirnames[:] = []
    return None


def search_local():
    """Ищем распакованную папку или zip в типичных местах. Возвращает ('dir'|'zip', путь) или None."""
    for base in ['.', '/content', '/content/drive/MyDrive', '/content/drive/My Drive']:
        if not os.path.isdir(base):
            continue
        for cand in (base, os.path.join(base, 'dataset')):
            if has_layout_dir(cand):
                return 'dir', os.path.abspath(cand)
        zips = sorted(glob.glob(os.path.join(base, '*.zip')),
                      key=lambda p: (os.path.basename(p) != 'archive.zip', p))   # archive.zip проверяем первым
        for z in zips:
            if has_layout_zip(z):
                return 'zip', os.path.abspath(z)
    if os.path.isdir('/kaggle/input'):
        p = walk_for_layout('/kaggle/input')
        if p:
            return 'dir', p
    return None


def locate_dataset(source='auto'):
    found = search_local()
    if found:
        return found
    if not IN_COLAB:
        raise FileNotFoundError(
            'Не нашёл данные. Положите archive.zip (или папку dataset/ с подпапками neg, neu, pos) '
            'рядом с ноутбуком и перезапустите ячейку.')
    print('Данные рядом с ноутбуком не найдены. Пробую способы для Colab:', source)
    steps = ['drive', 'kaggle', 'upload'] if source == 'auto' else [source]
    for step in steps:
        try:
            if step == 'drive':
                from google.colab import drive
                drive.mount('/content/drive')
                found = search_local()
                if not found:                       # архив мог лежать в подпапке Drive (на один уровень вглубь)
                    for z in glob.glob('/content/drive/MyDrive/*/archive.zip'):
                        if has_layout_zip(z):
                            found = ('zip', z)
                            break
                if not found:
                    print('  В Drive не найден archive.zip (ни в корне MyDrive, ни в подпапках 1-го уровня).')
            elif step == 'kaggle':
                import kagglehub
                p = kagglehub.dataset_download(KAGGLE_SLUG)
                print('  kagglehub скачал в', p)
                d = walk_for_layout(p)
                found = ('dir', d) if d else None
            elif step == 'upload':
                from google.colab import files
                print('  Выберите файл archive.zip в открывшемся окне ...')
                files.upload()                       # файл сохранится в текущую директорию (/content)
                found = search_local()
        except Exception as e:                       # пробуем следующий способ
            print(f'  Способ {step!r} не сработал: {type(e).__name__}: {e}')
            found = None
        if found:
            return found
    raise FileNotFoundError(
        'Не удалось получить данные. Загрузите archive.zip в /content или в корень Google Drive '
        'и перезапустите эту ячейку.')


kind, path = locate_dataset(DATA_SOURCE)

# чтение zip напрямую с Google Drive медленное (сетевая ФС) -> сначала копируем архив на локальный диск
if kind == 'zip' and path.startswith('/content/drive'):
    local_zip = '/content/archive_local.zip'
    if not os.path.exists(local_zip):
        print('Копирую архив с Drive на локальный диск Colab ...')
        shutil.copyfile(path, local_zip)
    path = local_zip

print(f'Источник данных: {kind} -> {path}')

In [ ]:
try:
    from tqdm.auto import tqdm
except ImportError:                       # tqdm нет — работаем без прогресс-бара
    def tqdm(x, **kw):
        return x


def iter_raw(kind, path):
    """Генератор (класс, имя файла без .txt, байты файла)."""
    if kind == 'dir':
        for lab in LABELS_ON_DISK:
            d = os.path.join(path, lab)
            for fn in sorted(os.listdir(d)):
                if fn.endswith('.txt'):
                    with open(os.path.join(d, fn), 'rb') as f:
                        yield lab, fn[:-4], f.read()
    else:
        with zipfile.ZipFile(path) as zf:
            for info in zf.infolist():
                if info.is_dir() or not info.filename.endswith('.txt'):
                    continue
                parts = info.filename.split('/')
                lab = parts[-2]
                if lab in LABELS_ON_DISK:
                    yield lab, parts[-1][:-4], zf.read(info)


def decode_bytes(raw):
    """UTF-8 (с учётом BOM); при ошибке — cp1251. Возвращает (текст, был_ли_fallback)."""
    try:
        return raw.decode('utf-8-sig'), False
    except UnicodeDecodeError:
        return raw.decode('cp1251', errors='replace'), True


t0 = time.time()
rows, n_fallback, RAW_BYTES = [], 0, 0
for lab, stem, raw in tqdm(iter_raw(kind, path), desc='Чтение отзывов'):
    text, fb = decode_bytes(raw)
    n_fallback += fb
    RAW_BYTES += len(raw)
    movie_id, _, review_no = stem.partition('-')       # имя файла: <id фильма>-<номер отзыва>
    rows.append((text, lab, movie_id, review_no))

df = pd.DataFrame(rows, columns=['text', 'label', 'movie_id', 'review_no'])
del rows
gc.collect()
print(f'Прочитано файлов: {len(df):,} за {time.time() - t0:.0f} c | файлов не в UTF-8 (cp1251-fallback): {n_fallback}')
print('Форма:', df.shape)

In [ ]:
display(df.head())
print('Колонки:', list(df.columns))
print()
df.info(memory_usage='deep')

### Классы и sanity check

Метка тональности берётся из **имени папки** (`neg` / `neu` / `pos`), так что нормализация не требуется.
Ниже — по два коротких примера каждого класса: глазами проверьте, что тексты соответствуют меткам.

In [ ]:
data = df
classes = [c for c in LABELS_ON_DISK if c in set(data['label'])]
label2id = {c: i for i, c in enumerate(classes)}
id2label = {i: c for c, i in label2id.items()}
NUM_CLASSES = len(classes)
print('Классы:', classes, '| label2id =', label2id)

for c in classes:
    print(f'\n--- {c} ---')
    for t in data.loc[data['label'] == c, 'text'].head(2):
        t = re.sub(r'\s+', ' ', t)
        print('  •', t[:200], '...' if len(t) > 200 else '')

## 1.1 Общая информация о данных
Количество отзывов, число классов и размер корпуса (символы, слова, объём в памяти).

In [ ]:
# Длины считаем один раз. Слова считаем через map(len(split)), а не .str.split() для всей колонки:
# так не создаётся ~100 млн строк-слов одновременно (экономия памяти).
data['n_chars'] = data['text'].str.len()
data['n_chars_strip'] = data['text'].map(lambda t: len(t.strip()) if isinstance(t, str) else 0)
data['n_words'] = data['text'].map(lambda t: len(t.split()) if isinstance(t, str) else 0)

n_reviews = len(data)
total_chars = int(data['n_chars'].sum())
total_words = int(data['n_words'].sum())

print(f'Количество отзывов:        {n_reviews:,}')
print(f'Количество классов:        {data["label"].nunique(dropna=True)}  -> {classes}')
print(f'Размер корпуса (символов): {total_chars:,}')
print(f'Размер корпуса (слов):     {total_words:,}')
print(f'Размер данных на диске:    {RAW_BYTES / 1e6:.1f} MB (UTF-8, сумма размеров всех файлов)')
print(f'Уникальных фильмов (по id в имени файла): {data["movie_id"].nunique():,}')

## 1.2 Пропуски и пустые строки в каждом классе
Считаем отдельно: пропуски (`NaN`) в тексте, «пустые» строки (только пробелы), слишком короткие (< 10 символов)
и строки без метки (они выделены в отдельную группу `<no label>`).
«Пустой файл» = файл, в котором после `strip()` не осталось символов.

In [ ]:
tmp = pd.DataFrame({
    'label': data['label'].fillna('<no label>'),
    'text_is_nan': data['text'].isna(),
    'empty_file': data['text'].notna() & data['n_chars_strip'].eq(0),
    f'short(<{MIN_CHARS})': data['n_chars_strip'].between(1, MIN_CHARS - 1),
})
missing_by_class = tmp.groupby('label').agg(
    total=('text_is_nan', 'size'),
    **{c: (c, 'sum') for c in tmp.columns if c != 'label'}
)
display(missing_by_class)
print('Пропуски по колонкам исходного датафрейма:')
display(data[['text', 'label', 'movie_id', 'review_no']].isna().sum().to_frame('n_missing').T)

## 1.3 Описательная статистика длины отзывов (символы и слова)
min, max, mean, median и квантили (25 / 50 / 75 / 95), в целом и по классам.

In [ ]:
eda = data.dropna(subset=['label'])      # для EDA берём только строки с меткой (длины уже посчитаны выше)

pcts = [0.25, 0.5, 0.75, 0.95]

def len_stats(s):
    d = s.describe(percentiles=pcts)
    out = pd.Series({'min': d['min'], 'max': d['max'], 'mean': d['mean'], 'median': s.median(),
                     'q25': d['25%'], 'q50': d['50%'], 'q75': d['75%'], 'q95': d['95%'], 'std': d['std']})
    return out.round(1)

print('=== Длина в символах: весь корпус ===')
display(len_stats(eda['n_chars']).to_frame('n_chars').T)
print('=== Длина в словах: весь корпус ===')
display(len_stats(eda['n_words']).to_frame('n_words').T)

print('=== Длина в символах по классам ===')
display(eda.groupby('label')['n_chars'].apply(len_stats).unstack().loc[classes])
print('=== Длина в словах по классам ===')
display(eda.groupby('label')['n_words'].apply(len_stats).unstack().loc[classes])

## 1.4 Уникальные значения целевой переменной и распределение по классам

In [ ]:
print('Уникальные значения метки (включая пропуски):', data['label'].unique().tolist())
class_counts = eda['label'].value_counts().reindex(classes)
class_share = (class_counts / class_counts.sum() * 100).round(2)
dist_table = pd.DataFrame({'count': class_counts, 'share, %': class_share})
display(dist_table)

## 1.5 Визуализации
**Barplot и pie chart** — распределение классов; **гистограммы** — распределение длин отзывов по классам
(в символах и в словах; правый «хвост» обрезан по 99-му перцентилю, чтобы выбросы не сжимали график).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
PAL = {c: PALETTE.get(c, '#6c757d') for c in classes}
colors = [PAL[c] for c in classes]

sns.barplot(x=class_counts.index, y=class_counts.values, hue=class_counts.index, palette=PAL, legend=False, ax=axes[0])
axes[0].set_title('Распределение классов (barplot)')
axes[0].set_xlabel('Класс тональности')
axes[0].set_ylabel('Количество отзывов')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=10)

axes[1].pie(class_counts.values, labels=[f'{c} ({s:.1f}%)' for c, s in zip(classes, class_share.values)],
            colors=colors, startangle=90, wedgeprops=dict(edgecolor='white'))
axes[1].set_title('Распределение классов (pie chart)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, col, xlabel in zip(axes, ['n_chars', 'n_words'], ['Длина отзыва, символы', 'Длина отзыва, слова']):
    clip = eda[col].quantile(0.99)
    sns.histplot(data=eda.loc[eda[col] <= clip, [col, 'label']], x=col, hue='label', hue_order=classes, palette=PAL,
                 bins=50, stat='density', common_norm=False, element='step', alpha=0.25, ax=ax)
    ax.set_title(f'Распределение длин отзывов по классам ({xlabel.split(",")[1].strip()})')
    ax.set_xlabel(xlabel + ' (обрезано по 99-му перцентилю)')
    ax.set_ylabel('Плотность')
plt.tight_layout()
plt.show()

## 1.6 Вывод о дисбалансе классов и его влиянии на обучение
Ниже числа считаются автоматически по вашим данным, а под ними — общие соображения.

In [ ]:
most, least = class_counts.idxmax(), class_counts.idxmin()
imb_ratio = class_counts.max() / class_counts.min()
display(Markdown(
    f'**Самый частый класс:** `{most}` ({class_counts[most]:,} отзывов, {class_share[most]:.1f}%). '
    f'**Самый редкий:** `{least}` ({class_counts[least]:,}, {class_share[least]:.1f}%). '
    f'**Отношение max/min = {imb_ratio:.1f}×.** '
    + ('Дисбаланс **выраженный**.' if imb_ratio >= 3 else 'Дисбаланс **умеренный**.' if imb_ratio >= 1.5
       else 'Классы **почти сбалансированы**.')
))

**Как дисбаланс влияет на обучение (общие соображения):**
- Модель, минимизирующая обычную кросс-энтропию, «выгодно» смещается в сторону частых классов: их ошибки дают
  больший суммарный вклад в loss. Редкий класс распознаётся хуже (низкий recall), при этом **accuracy остаётся
  обманчиво высокой**.
- Поэтому основной метрикой мы берём **macro F1** (усредняет F1 по классам с равным весом), а accuracy
  используем как вспомогательную.
- Класс `neu` по смыслу «размытый» (граница с позитивом/негативом нечёткая), поэтому даже будучи не самым редким
  он обычно получает самый низкий F1. Здесь доминирует `pos`, самый редкий — `neg`.
- На гистограммах видно, что распределения длин по классам практически совпадают, поэтому длина — слабый признак
  тональности. Зато она важна для выбора `max_len` (LSTM) и `max_length` (BERT): отзывы длинные (медиана — порядка
  300 слов), и при `max_length = 256` значительная часть текста будет обрезана.
- Стратегия борьбы с дисбалансом описана в п. 2.5: взвешенная кросс-энтропия + стратифицированные разбиения.

# 2. Предобработка данных

## 2.1 Удаление пустых и слишком коротких отзывов
Последовательно удаляем: строки без метки / с пропуском в тексте, отзывы короче 10 символов (после `strip`)
и **точные дубликаты текста** (несколько десятков штук; иначе один и тот же отзыв мог бы попасть и в train, и в test).
Для экономии времени и памяти нормализацию пробелов (`\s+` → пробел) применяем позже — только к тем отзывам,
которые реально попадают в обучение.

In [ ]:
def basic_clean(t):
    """Схлопываем повторяющиеся пробелы и переводы строк."""
    return re.sub(r'\s+', ' ', str(t)).strip()


steps = [('Всего файлов', len(data))]
clean = data.dropna(subset=['label', 'text'])
steps.append(('после удаления NaN (текст / метка)', len(clean)))
clean = clean[clean['n_chars_strip'] >= MIN_CHARS]
steps.append((f'после удаления пустых и коротких (< {MIN_CHARS} симв.)', len(clean)))
clean = clean[~clean.duplicated(subset='text', keep='first')]
steps.append(('после удаления точных дубликатов текста', len(clean)))

clean = clean[['text', 'label', 'movie_id']].reset_index(drop=True)
clean['y'] = clean['label'].map(label2id).astype(int)

steps_df = pd.DataFrame(steps, columns=['шаг', 'осталось отзывов'])
steps_df['удалено'] = steps_df['осталось отзывов'].shift(1) - steps_df['осталось отзывов']
display(steps_df)
display(clean['label'].value_counts().reindex(classes).to_frame('count'))

## 2.4 Разделение train / validation / test = 70 / 15 / 15 со стратификацией
*Порядок изложения:* разбиение делаем **до** токенизации и построения словаря (пп. 2.2–2.3), чтобы словарь LSTM
строился только по train (без утечки информации из val/test).

Сначала делим весь очищенный корпус на 70/15/15 (`train_test_split` со `stratify`; делим индексы, а не тексты — так не копируем ~800 МБ текста). Затем из каждой части берём
**стратифицированную подвыборку** (в сумме `SUBSET_TOTAL` примеров, те же 70/15/15) — на ней обучаются и сравниваются
**все три модели**, чтобы сравнение было честным и BERT-эксперименты быстро считались.

In [ ]:
y_all = clean['y'].values
idx_all = np.arange(len(clean))
tr_idx, tmp_idx = train_test_split(idx_all, test_size=0.30, stratify=y_all, random_state=SEED)
va_idx, te_idx = train_test_split(tmp_idx, test_size=0.50, stratify=y_all[tmp_idx], random_state=SEED)
print(f'Полные разбиения: train={len(tr_idx):,} | val={len(va_idx):,} | test={len(te_idx):,}')


def stratified_subsample_idx(idx, n):
    if n is None or n >= len(idx):
        return idx
    sub, _ = train_test_split(idx, train_size=n, stratify=y_all[idx], random_state=SEED)
    return sub


if SUBSET_TOTAL is None:
    n_tr = n_va = n_te = None
else:
    n_tr, n_va = int(0.70 * SUBSET_TOTAL), int(0.15 * SUBSET_TOTAL)
    n_te = SUBSET_TOTAL - n_tr - n_va

train_df = clean.iloc[stratified_subsample_idx(tr_idx, n_tr)].reset_index(drop=True)
val_df = clean.iloc[stratified_subsample_idx(va_idx, n_va)].reset_index(drop=True)
test_df = clean.iloc[stratified_subsample_idx(te_idx, n_te)].reset_index(drop=True)

# нормализация пробелов — только для реально используемых отзывов
for d in (train_df, val_df, test_df):
    d['text'] = d['text'].map(basic_clean)

print(f'Подвыборка для экспериментов: train={len(train_df):,} | val={len(val_df):,} | test={len(test_df):,}')

split_table = pd.DataFrame({
    'train': train_df['label'].value_counts(normalize=True).reindex(classes) * 100,
    'val': val_df['label'].value_counts(normalize=True).reindex(classes) * 100,
    'test': test_df['label'].value_counts(normalize=True).reindex(classes) * 100,
    'full corpus': clean['label'].value_counts(normalize=True).reindex(classes) * 100,
}).round(2)
print('Доли классов, % (стратификация сохраняет распределение):')
display(split_table)

# отзывы разбиваются случайно, а не по фильмам -> о части фильмов есть отзывы и в train, и в test
movie_overlap = float(test_df['movie_id'].isin(set(train_df['movie_id'])).mean())
print(f'Доля test-отзывов о фильмах, которые встречаются и в train: {movie_overlap * 100:.1f}%')

## 2.5 Веса классов и обоснование подхода к дисбалансу
Считаем веса `compute_class_weight('balanced')` **по train-подвыборке**: `w_c = N / (K · n_c)`.
Веса передаём в `CrossEntropyLoss(weight=...)` для **всех** моделей.

**Почему именно взвешенный loss:**
- **Не меняет данные** и не увеличивает время эпохи (в отличие от oversampling, который к тому же дублирует
  примеры и повышает риск переобучения, особенно у BERT).
- **Не выбрасывает информацию** (в отличие от undersampling, особенно на небольшой подвыборке).
- Прямо оптимизирует «равноправие» классов, что согласуется с целевой метрикой macro F1.
- Стратификация гарантирует одинаковые пропорции классов в train/val/test, а значит, метрики на val/test
  сопоставимы.

Минусы: иногда падает precision редкого класса и слегка страдает accuracy; при сильном дисбалансе веса могут
давать нестабильные градиенты (тогда помогает сглаживание, например, корень из весов). Альтернативы (focal loss,
oversampling, аугментация) — в разделе выводов.

In [ ]:
class_weights_np = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_df['y'].values)
class_weights = torch.tensor(class_weights_np, dtype=torch.float)
cw_dict = {c: round(float(w), 2) for c, w in zip(classes, class_weights_np)}
cw_table = pd.DataFrame({
    'class': classes,
    'n_train': train_df['y'].value_counts().reindex(range(NUM_CLASSES)).values,
    'weight': class_weights_np.round(3),
})
display(cw_table)

## 2.2 Токенизация для BERT
Используем токенизатор `DeepPavlov/rubert-base-cased` (при недоступности — `bert-base-multilingual-cased`).
Для BERT текст оставляем «как есть» (регистр и пунктуацию не трогаем — модель `cased`, а WordPiece сам
разбирается с редкими словами). Смотрим, какая доля отзывов не помещается в 256 токенов.

In [ ]:
try:
    BERT_NAME = 'DeepPavlov/rubert-base-cased'
    tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
except Exception as e:
    print('rubert недоступен, переключаемся на multilingual:', e)
    BERT_NAME = 'bert-base-multilingual-cased'
    tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
print('Модель/токенизатор:', BERT_NAME)

example = train_df['text'].iloc[0]
print('\nПример текста:', example[:150], '...')
print('Токены:', tokenizer.tokenize(example)[:25], '...')

# длина в токенах БЕЗ усечения (verbose=False убирает предупреждение о длине > 512)
tok_lens = np.array([len(x) for x in tokenizer(train_df['text'].tolist(), truncation=False, verbose=False)['input_ids']])
trunc_share_bert = float((tok_lens > BERT_MAX_LEN).mean())
print(f'\nДлина в токенах (train, без усечения): median={np.median(tok_lens):.0f}, '
      f'q95={np.percentile(tok_lens, 95):.0f}, max={tok_lens.max()}')
print(f'Доля отзывов длиннее {BERT_MAX_LEN} токенов (будут усечены): {trunc_share_bert * 100:.1f}%')

q99 = np.percentile(tok_lens, 99)
plt.figure(figsize=(7, 4))
sns.histplot(tok_lens[tok_lens <= q99], bins=50, color='#4c72b0')
plt.axvline(BERT_MAX_LEN, color='red', ls='--', label=f'max_length = {BERT_MAX_LEN}')
plt.title('Длина отзывов в токенах BERT (train, обрезано по 99-му перцентилю)')
plt.xlabel('Число токенов')
plt.ylabel('Количество отзывов')
plt.legend()
plt.show()

## 2.3 Предобработка для LSTM/GRU: нижний регистр, очистка, словарь
- приводим к нижнему регистру, `ё → е`, удаляем ссылки и HTML-теги;
- оставляем только буквы (кириллица/латиница), цифры и знаки `!` `?` (они несут эмоциональную окраску и
  выделяются отдельными токенами);
- словарь строим **только по train**: спец-токены `<pad>` (индекс 0) и `<unk>` (индекс 1), затем слова с
  частотой ≥ `MIN_FREQ`, максимум `MAX_VOCAB` слов.

In [ ]:
def clean_for_rnn(t):
    t = str(t).lower().replace('ё', 'е')
    t = re.sub(r'https?://\S+|www\.\S+', ' ', t)
    t = re.sub(r'<[^>]+>', ' ', t)
    t = re.sub(r'[^а-яa-z0-9!?\s]', ' ', t)
    t = re.sub(r'([!?])', r' \1 ', t)
    return re.sub(r'\s+', ' ', t).strip()

for d in (train_df, val_df, test_df):
    d['tokens'] = d['text'].map(lambda s: clean_for_rnn(s).split())

MIN_FREQ, MAX_VOCAB = 2, 30000
counter = Counter(tok for toks in train_df['tokens'] for tok in toks)
itos = ['<pad>', '<unk>'] + [w for w, c in counter.most_common(MAX_VOCAB) if c >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(itos)}
PAD_IDX, UNK_IDX = stoi['<pad>'], stoi['<unk>']
VOCAB_SIZE = len(itos)

total_tokens = sum(counter.values())
covered = sum(c for w, c in counter.items() if w in stoi)
print(f'Размер словаря: {VOCAB_SIZE:,} (уникальных слов в train: {len(counter):,})')
print(f'Покрытие токенов train словарём: {covered / total_tokens * 100:.1f}%')
print('Пример:', train_df['tokens'].iloc[0][:15])

trunc_share_rnn = float((train_df['tokens'].map(len) > RNN_MAX_LEN).mean())
print(f'Доля train-отзывов длиннее {RNN_MAX_LEN} слов (будут усечены в LSTM/GRU): {trunc_share_rnn * 100:.1f}%')

# 3. Базовая модель: двунаправленная LSTM/GRU (PyTorch)

Сначала — общая обучающая «обвязка», которую мы будем использовать и для LSTM, и для BERT:
- все модели принимают словарь тензоров `model(**inputs) → logits`, поэтому цикл обучения един;
- метрики: accuracy и **macro F1**; loss — `CrossEntropyLoss` с весами классов;
- **early stopping по validation macro F1** с восстановлением лучших весов;
- функции построения графиков loss/F1 по эпохам и вывода `classification_report` + confusion matrix на test.

In [ ]:
def count_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable_only))


def run_epoch(model, loader, criterion, optimizer=None, scheduler=None, scaler=None,
              freeze_encoder=False, use_amp=False, max_grad_norm=1.0):
    """Одна эпоха обучения (если передан optimizer) или оценки. Возвращает loss, acc, macro-F1 и предсказания."""
    is_train = optimizer is not None
    model.train(is_train)
    if is_train and freeze_encoder:
        model.hf.base_model.eval()        # замороженный энкодер: отключаем dropout в нём
    total_loss, n_seen = 0.0, 0
    ys, ps = [], []
    with torch.set_grad_enabled(is_train):
        for inputs, labels in loader:
            inputs = {k: v.to(device) for k, v in inputs.items()}
            labels = labels.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(**inputs)
            loss = criterion(logits.float(), labels)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                if scheduler is not None:
                    scheduler.step()
            bs = labels.size(0)
            total_loss += loss.item() * bs
            n_seen += bs
            ys.append(labels.cpu().numpy())
            ps.append(logits.argmax(dim=1).cpu().numpy())
    y_true, y_pred = np.concatenate(ys), np.concatenate(ps)
    return {
        'loss': total_loss / n_seen,
        'acc': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, average='macro'),
        'y_true': y_true, 'y_pred': y_pred,
    }


def fit(model, train_loader, val_loader, criterion, optimizer, scheduler=None, epochs=10, patience=3,
        name='model', freeze_encoder=False, use_amp=False):
    """Обучение с early stopping по validation macro F1; в конце восстанавливаются лучшие веса."""
    history = {k: [] for k in ['train_loss', 'val_loss', 'train_f1', 'val_f1', 'train_acc', 'val_acc']}
    try:
        scaler = torch.amp.GradScaler('cuda', enabled=use_amp)     # torch >= 2.3
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    best_f1, best_state, best_epoch, bad_epochs = -1.0, None, 0, 0
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        t_ep = time.time()
        tr = run_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, freeze_encoder, use_amp)
        va = run_epoch(model, val_loader, criterion, use_amp=use_amp)
        for key, res in (('train', tr), ('val', va)):
            history[f'{key}_loss'].append(res['loss'])
            history[f'{key}_f1'].append(res['f1'])
            history[f'{key}_acc'].append(res['acc'])
        improved = va['f1'] > best_f1 + 1e-4
        if improved:
            best_f1, best_epoch, bad_epochs = va['f1'], epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
        print(f'[{name}] epoch {epoch:02d}/{epochs} | train loss {tr["loss"]:.4f} F1 {tr["f1"]:.4f} | '
              f'val loss {va["loss"]:.4f} F1 {va["f1"]:.4f} acc {va["acc"]:.4f} | '
              f'{time.time() - t_ep:.0f}s{"  *" if improved else ""}')
        if bad_epochs >= patience:
            print(f'[{name}] early stopping: val F1 не улучшался {patience} эпох подряд.')
            break
    model.load_state_dict(best_state)
    info = {'best_epoch': best_epoch, 'best_val_f1': best_f1, 'train_time_s': time.time() - t0,
            'epochs_run': len(history['val_f1'])}
    print(f'[{name}] лучшая эпоха: {best_epoch} (val macro F1 = {best_f1:.4f}), время: {info["train_time_s"]:.0f}s')
    return history, info


def plot_history(history, info, title):
    epochs = np.arange(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history['train_loss'], 'o-', label='train loss')
    axes[0].plot(epochs, history['val_loss'], 's-', label='val loss')
    axes[0].set_title(f'{title}: loss по эпохам')
    axes[0].set_xlabel('Эпоха'); axes[0].set_ylabel('Loss')
    axes[1].plot(epochs, history['train_f1'], 'o-', label='train macro F1')
    axes[1].plot(epochs, history['val_f1'], 's-', label='val macro F1')
    axes[1].set_title(f'{title}: macro F1 по эпохам')
    axes[1].set_xlabel('Эпоха'); axes[1].set_ylabel('Macro F1')
    for ax in axes:
        ax.axvline(info['best_epoch'], color='gray', ls='--', alpha=0.7, label=f'лучшая эпоха = {info["best_epoch"]}')
        ax.set_xticks(epochs)
        ax.legend()
    plt.tight_layout()
    plt.show()


results = {}   # сюда складываем итоговые метрики каждой модели


def evaluate_test(model, test_loader, criterion, name, info, history, use_amp=False, trainable_params=None):
    res = run_epoch(model, test_loader, criterion, use_amp=use_amp)
    y_true, y_pred = res['y_true'], res['y_pred']
    rep = classification_report(y_true, y_pred, target_names=classes, digits=4, output_dict=True, zero_division=0)
    print(f'===== {name}: TEST =====')
    print(classification_report(y_true, y_pred, target_names=classes, digits=4, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=axes[0])
    axes[0].set_title(f'{name}: confusion matrix (counts)')
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=classes, yticklabels=classes,
                vmin=0, vmax=1, ax=axes[1])
    axes[1].set_title(f'{name}: confusion matrix (по строкам, recall)')
    for ax in axes:
        ax.set_xlabel('Предсказано'); ax.set_ylabel('Истина')
    plt.tight_layout()
    plt.show()
    results[name] = {
        'accuracy': res['acc'], 'macro_f1': res['f1'], 'weighted_f1': rep['weighted avg']['f1-score'],
        'per_class_f1': {c: rep[c]['f1-score'] for c in classes},
        'trainable_params': trainable_params if trainable_params is not None else count_params(model, True),
        'train_time_s': info['train_time_s'], 'best_epoch': info['best_epoch'],
        'history': history, 'best_val_f1': info['best_val_f1'],
    }

### 3.1 Гиперпараметры и подготовка данных для LSTM/GRU
Все гиперпараметры зафиксированы в словаре `HP_RNN` и выводятся ниже. Тип ячейки переключается параметром
`rnn_type` (`'LSTM'` или `'GRU'`).

In [ ]:
HP_RNN = dict(
    rnn_type='LSTM',       # 'LSTM' или 'GRU'
    emb_dim=128,           # размерность эмбеддингов (случайная инициализация)
    hidden_size=128,
    num_layers=2,          # по ДЗ: минимум 2 слоя
    bidirectional=True,    # двунаправленная
    dropout=0.3,
    lr=1e-3,
    batch_size=64,
    max_len=RNN_MAX_LEN,   # максимальная длина последовательности (в словах)
    num_epochs=15,
    patience=3,            # early stopping по val macro F1
)
print('Гиперпараметры LSTM/GRU:')
display(pd.Series(HP_RNN).to_frame('value'))


class RNNDataset(Dataset):
    def __init__(self, token_lists, labels, max_len):
        self.ids = [[stoi.get(t, UNK_IDX) for t in toks][:max_len] for toks in token_lists]
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.ids[i], self.labels[i]


def collate_rnn(batch):
    seqs, ys = zip(*batch)
    lengths = torch.tensor([max(len(s), 1) for s in seqs], dtype=torch.long)
    x = torch.full((len(seqs), int(lengths.max())), PAD_IDX, dtype=torch.long)
    for i, s in enumerate(seqs):
        if len(s) > 0:
            x[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    return {'input_ids': x, 'lengths': lengths}, torch.tensor(ys, dtype=torch.long)


def make_loader(ds, batch_size, shuffle, collate_fn):
    g = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn, generator=g)


rnn_train_dl = make_loader(RNNDataset(train_df['tokens'], train_df['y'], HP_RNN['max_len']), HP_RNN['batch_size'], True, collate_rnn)
rnn_val_dl = make_loader(RNNDataset(val_df['tokens'], val_df['y'], HP_RNN['max_len']), 128, False, collate_rnn)
rnn_test_dl = make_loader(RNNDataset(test_df['tokens'], test_df['y'], HP_RNN['max_len']), 128, False, collate_rnn)
print('Батчей в train/val/test:', len(rnn_train_dl), len(rnn_val_dl), len(rnn_test_dl))

### 3.2 Архитектура
`Embedding (случайная инициализация, padding_idx=<pad>)` → dropout → **2-слойная двунаправленная LSTM/GRU**
(используем `pack_padded_sequence`, чтобы паддинг не влиял на скрытые состояния) → конкатенация последних
скрытых состояний прямого и обратного проходов верхнего слоя → dropout → линейный классификатор.

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_layers, num_classes, dropout,
                 rnn_type='LSTM', bidirectional=True, pad_idx=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        rnn_cls = nn.LSTM if rnn_type == 'LSTM' else nn.GRU
        self.rnn = rnn_cls(emb_dim, hidden_size, num_layers=num_layers, batch_first=True,
                           bidirectional=bidirectional, dropout=dropout if num_layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.bidirectional = bidirectional
        self.fc = nn.Linear(hidden_size * (2 if bidirectional else 1), num_classes)

    def forward(self, input_ids, lengths):
        x = self.drop(self.emb(input_ids))
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.rnn(packed)
        if isinstance(h, tuple):          # LSTM возвращает (h_n, c_n)
            h = h[0]
        h = torch.cat([h[-2], h[-1]], dim=1) if self.bidirectional else h[-1]
        return self.fc(self.drop(h))


set_seed()
rnn_model = RNNClassifier(VOCAB_SIZE, HP_RNN['emb_dim'], HP_RNN['hidden_size'], HP_RNN['num_layers'],
                          NUM_CLASSES, HP_RNN['dropout'], HP_RNN['rnn_type'], HP_RNN['bidirectional'], PAD_IDX).to(device)
print(rnn_model)
print(f'\nОбучаемых параметров: {count_params(rnn_model, True):,}')

### 3.3 Обучение
`CrossEntropyLoss` с весами классов, оптимизатор `Adam`, early stopping по validation macro F1.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
rnn_optimizer = torch.optim.Adam(rnn_model.parameters(), lr=HP_RNN['lr'])
rnn_name = f'{HP_RNN["rnn_type"]} (BiLSTM, {HP_RNN["num_layers"]} слоя)' if HP_RNN['rnn_type'] == 'LSTM' \
    else f'GRU (BiGRU, {HP_RNN["num_layers"]} слоя)'

rnn_history, rnn_info = fit(rnn_model, rnn_train_dl, rnn_val_dl, criterion, rnn_optimizer, scheduler=None,
                            epochs=HP_RNN['num_epochs'], patience=HP_RNN['patience'], name=rnn_name)

### 3.4 Графики обучения и оценка на test

In [ ]:
plot_history(rnn_history, rnn_info, rnn_name)
evaluate_test(rnn_model, rnn_test_dl, criterion, rnn_name, rnn_info, rnn_history)

# 4. Модель на основе BERT (HuggingFace transformers)

Используем `AutoTokenizer` + `AutoModelForSequenceClassification` (голова классификации — линейный слой поверх
pooled-выхода `[CLS]`). Параметры эксперимента:
- `max_length = 256`, `truncation=True` (длинные отзывы обрезаются), динамический паддинг внутри батча;
- **AdamW + линейный warmup-шедулер** (`get_linear_schedule_with_warmup`, 10% шагов на разогрев);
- loss — та же взвешенная `CrossEntropyLoss`;
- при наличии GPU используется смешанная точность (fp16) для ускорения;
- два режима: **(a) Frozen encoder** — обучается только классификационная голова (`requires_grad=False` для всех
  параметров базовой модели); **(b) Fine-tuning** — обучается вся модель, `lr` в диапазоне 2e-5…5e-5.

Обе модели обучаются на одной и той же подвыборке и оцениваются на одном и том же test.

> При загрузке модели HuggingFace напишет, что часть весов (`classifier.weight`, `classifier.bias`) «newly initialized» —
> это ожидаемо: классификационная голова создаётся с нуля и обучается на нашей задаче. Предупреждение про `HF_TOKEN`
> в Colab тоже можно игнорировать (модель публичная).

In [ ]:
HP_BERT = dict(
    model_name=BERT_NAME,
    max_length=BERT_MAX_LEN,
    batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.01,
    frozen=dict(lr=1e-3, epochs=5, patience=2),     # обучается только голова -> lr можно брать больше
    finetune=dict(lr=3e-5, epochs=3, patience=2),   # по ДЗ: 2e-5 ... 5e-5
)
print('Гиперпараметры BERT:')
display(pd.Series({k: str(v) for k, v in HP_BERT.items()}).to_frame('value'))


class BertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.enc = tokenizer(list(texts), truncation=True, max_length=max_length)   # без паддинга: дополним в батче
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {k: self.enc[k][i] for k in self.enc.keys()}, self.labels[i]


def collate_bert(batch):
    feats, ys = zip(*batch)
    padded = tokenizer.pad(list(feats), return_tensors='pt')
    return dict(padded), torch.tensor(ys, dtype=torch.long)


class HFWrapper(nn.Module):
    """Обёртка: forward(**inputs) -> logits (единый интерфейс с LSTM)."""
    def __init__(self, hf_model):
        super().__init__()
        self.hf = hf_model

    def forward(self, **inputs):
        return self.hf(**inputs).logits


bert_train_dl = make_loader(BertDataset(train_df['text'], train_df['y'], tokenizer, BERT_MAX_LEN), HP_BERT['batch_size'], True, collate_bert)
bert_val_dl = make_loader(BertDataset(val_df['text'], val_df['y'], tokenizer, BERT_MAX_LEN), 32, False, collate_bert)
bert_test_dl = make_loader(BertDataset(test_df['text'], test_df['y'], tokenizer, BERT_MAX_LEN), 32, False, collate_bert)
print('Батчей в train/val/test:', len(bert_train_dl), len(bert_val_dl), len(bert_test_dl))

use_amp = device.type == 'cuda'


def build_bert(freeze_encoder):
    set_seed()
    hf = AutoModelForSequenceClassification.from_pretrained(
        BERT_NAME, num_labels=NUM_CLASSES, id2label=id2label, label2id=label2id)
    if freeze_encoder:
        for p in hf.base_model.parameters():     # замораживаем весь энкодер (эмбеддинги + слои + pooler)
            p.requires_grad = False
    return HFWrapper(hf).to(device)


def train_bert(freeze_encoder, name, cfg):
    model = build_bert(freeze_encoder)
    n_train, n_all = count_params(model, True), count_params(model, False)
    print(f'{name}: обучаемых параметров {n_train:,} из {n_all:,} ({n_train / n_all * 100:.2f}%)')
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=cfg['lr'], weight_decay=HP_BERT['weight_decay'])
    total_steps = len(bert_train_dl) * cfg['epochs']
    scheduler = get_linear_schedule_with_warmup(optimizer, int(HP_BERT['warmup_ratio'] * total_steps), total_steps)
    history, info = fit(model, bert_train_dl, bert_val_dl, criterion, optimizer, scheduler,
                        epochs=cfg['epochs'], patience=cfg['patience'], name=name,
                        freeze_encoder=freeze_encoder, use_amp=use_amp)
    plot_history(history, info, name)
    evaluate_test(model, bert_test_dl, criterion, name, info, history, use_amp=use_amp, trainable_params=n_train)
    del model, optimizer, scheduler
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

## 4.a Frozen encoder: обучается только классификационная голова
Веса BERT не меняются, поэтому модель использует «общие» языковые представления, не подстроенные под задачу
тональности. Обычно это заметно слабее полного дообучения, но существенно дешевле по числу обучаемых параметров.

In [ ]:
train_bert(True, 'BERT frozen', HP_BERT['frozen'])

## 4.b Fine-tuning: дообучается вся модель
`lr = 3e-5`, AdamW с weight decay, линейный warmup (10% шагов) и линейное затухание lr, clip градиента = 1.0.

In [ ]:
train_bert(False, 'BERT fine-tuned', HP_BERT['finetune'])

# 5. Сравнение моделей и выводы

## 5.1 Сводная таблица метрик на test

In [ ]:
summary = pd.DataFrame({
    name: {
        'Accuracy': r['accuracy'],
        'Macro F1': r['macro_f1'],
        'Weighted F1': r['weighted_f1'],
        **{f'F1 [{c}]': r['per_class_f1'][c] for c in classes},
        'Best val macro F1': r['best_val_f1'],
        'Обучаемых параметров': r['trainable_params'],
        'Время обучения, с': r['train_time_s'],
        'Лучшая эпоха': r['best_epoch'],
    } for name, r in results.items()
}).T
fmt = {c: '{:.4f}' for c in summary.columns if c not in ('Обучаемых параметров', 'Время обучения, с', 'Лучшая эпоха')}
fmt.update({'Обучаемых параметров': '{:,.0f}', 'Время обучения, с': '{:.0f}', 'Лучшая эпоха': '{:.0f}'})
try:
    display(summary.style.format(fmt).highlight_max(subset=['Accuracy', 'Macro F1', 'Weighted F1'], color='#c6efce'))
except Exception:   # если нет jinja2 — выводим обычную таблицу
    display(summary.round(4))

summary.to_csv('results_summary.csv')       # сохраняем таблицу (в Colab: папка Files слева)
print('Таблица сохранена в results_summary.csv')

## 5.2 Сравнительный barplot: accuracy и macro F1

In [ ]:
plot_df = summary[['Accuracy', 'Macro F1']].reset_index().rename(columns={'index': 'Модель'}).melt(
    id_vars='Модель', var_name='Метрика', value_name='Значение')
plt.figure(figsize=(9, 5))
ax = sns.barplot(data=plot_df, x='Модель', y='Значение', hue='Метрика', palette=['#4c72b0', '#dd8452'])
for cont in ax.containers:
    ax.bar_label(cont, fmt='%.3f', padding=2)
ax.set_ylim(0, min(1.0, plot_df['Значение'].max() + 0.12))
ax.set_title('Сравнение моделей на test: accuracy и macro F1')
ax.set_xlabel('Модель')
ax.set_ylabel('Значение метрики')
plt.xticks(rotation=10)
plt.legend(title='Метрика')
plt.tight_layout()
plt.show()

In [ ]:
# Дополнительно: динамика validation macro F1 у всех моделей
plt.figure(figsize=(8, 4.5))
for name, r in results.items():
    plt.plot(np.arange(1, len(r['history']['val_f1']) + 1), r['history']['val_f1'], 'o-', label=name)
plt.title('Validation macro F1 по эпохам')
plt.xlabel('Эпоха')
plt.ylabel('Validation macro F1')
plt.legend()
plt.tight_layout()
plt.show()

## 5.3 Выводы
Ответы на вопросы задания, подставленные из полученных результатов (ячейка ниже), а затем — развёрнутые
рассуждения.

In [ ]:
best_name = summary['Macro F1'].idxmax()
worst_name = summary['Macro F1'].idxmin()
cls_f1 = pd.Series(results[best_name]['per_class_f1'])
hardest_cls, easiest_cls = cls_f1.idxmin(), cls_f1.idxmax()

display(Markdown(f"""
**Ответы на вопросы задания (числа получены при этом запуске):**

1. **Самый частый / самый редкий класс:** `{most}` ({class_share[most]:.1f}%) / `{least}` ({class_share[least]:.1f}%); отношение max/min ≈ {imb_ratio:.1f}×.
2. **Как обрабатывали дисбаланс:** стратифицированное разбиение 70/15/15 (доли классов одинаковы в train/val/test),
   веса классов `compute_class_weight('balanced')` в `CrossEntropyLoss` ({cw_dict}) и оценка по **macro F1**, а не только по accuracy.
3. **Лучшая модель по macro F1:** **{best_name}** — macro F1 = **{summary.loc[best_name, 'Macro F1']:.4f}**, accuracy = {summary.loc[best_name, 'Accuracy']:.4f}.
   Худшая: {worst_name} (macro F1 = {summary.loc[worst_name, 'Macro F1']:.4f}). Разрыв лучшей и худшей: {summary['Macro F1'].max() - summary['Macro F1'].min():+.4f}.
4. **Самый трудный класс** для лучшей модели: `{hardest_cls}` (F1 = {cls_f1[hardest_cls]:.3f}); самый лёгкий: `{easiest_cls}` (F1 = {cls_f1[easiest_cls]:.3f}).
5. **Особенности данных, влияющие на результат:** {trunc_share_bert * 100:.0f}% train-отзывов длиннее {BERT_MAX_LEN} токенов и усекаются для BERT
   ({trunc_share_rnn * 100:.0f}% длиннее {RNN_MAX_LEN} слов и усекаются для LSTM); {movie_overlap * 100:.0f}% test-отзывов относятся к фильмам, которые есть и в train.
"""))

**Почему BERT (обычно) обгоняет LSTM/GRU.**
- **Предобучение.** BERT уже «знает» язык: он предобучен на огромных корпусах и несёт контекстные представления слов,
  а наш LSTM с случайными эмбеддингами вынужден выучить и лексику, и тональность из нескольких тысяч отзывов.
  На малых выборках это главное преимущество.
- **Attention вместо последовательного сжатия.** Self-attention даёт каждому токену прямой доступ ко всем остальным
  (короткий путь для градиента, учёт дальних зависимостей и отрицаний вроде «не …, но …»), тогда как RNN сжимает
  контекст в скрытое состояние фиксированного размера и хуже работает на длинных отзывах.
- **Контекстные эмбеддинги и подслова.** Одно слово получает разные представления в разных контекстах, а WordPiece
  почти не даёт `<unk>` (в LSTM все слова вне словаря превращаются в `<unk>`).
- **Frozen vs. fine-tuning.** Замороженный энкодер даёт представления, не настроенные на тональность, поэтому обычно
  заметно слабее fine-tuning; при этом он часто сопоставим с LSTM или лучше, а обучение головы стоит очень мало
  параметров. Полное дообучение подстраивает все слои под задачу и, как правило, даёт лучший результат — ценой
  большего времени, памяти и риска переобучения.
- Если в вашем запуске картина иная (например, LSTM оказался близок к BERT frozen), это допустимо: на малых
  подвыборках и коротких обучениях разница определяется шумом, гиперпараметрами и качеством меток.

**Влияние дисбаланса классов.** Без весов модели склонны предсказывать частые классы, что завышает accuracy и
«прячет» провал по редкому классу. Взвешенный loss смещает баланс precision/recall в пользу редких классов,
обычно повышая macro F1 (иногда ценой небольшого падения accuracy). Смотрите на per-class F1 и confusion matrix:
основная путаница чаще всего происходит между нейтральным классом и соседними (позитивным/негативным).

**Ограничения эксперимента.**
- Все модели обучены и сравнивались на **небольшой подвыборке** (по умолчанию 5 000 отзывов: 3 500 / 750 / 750) ради скорости;
  на полном корпусе абсолютные метрики (особенно у LSTM), как правило, будут выше, а разрыв может измениться.
- **Один seed и один сплит** — нет доверительных интервалов; при 750 test-примерах разница в 1–2 п.п. F1 может быть шумом.
- Гиперпараметры не подбирались (фиксированный набор); у frozen-режима lr выбран эвристически.
- **Обрезка до 256 токенов теряет большую часть текста.** Рецензии на Кинопоиске длинные (медиана — порядка 300 слов,
  то есть заведомо больше 256 токенов), поэтому BERT видит лишь начало рецензии, а вывод автора («в итоге фильм ...»)
  часто стоит в конце. Точная доля усечённых отзывов приведена в п. 2.2 и в ответе №5 выше. То же относится и к LSTM (`max_len = 256` слов).
- **Разбиение случайное по отзывам, а не по фильмам:** часть test-отзывов написана о фильмах, отзывы на которые есть и в train
  (доля указана выше), что может слегка завышать метрики. Более строгий вариант — `GroupShuffleSplit` по `movie_id`.
- **Природа меток.** Класс (`pos/neu/neg`) — это категория рецензии, скорее всего выбранная самим автором, а не независимая
  разметка: тон текста и выбранная категория могут расходиться (особенно для `neu`).
- LSTM обучался с эмбеддингами со случайной инициализацией; предобученные fastText/word2vec могли бы сократить разрыв.

**Идеи для улучшения.**
- *Длинные тексты:* усечение «голова + хвост» (первые ~128 и последние ~128 токенов), либо разбиение отзыва на окна по 256 токенов
  с усреднением логитов; модели с длинным контекстом.
- *Данные:* обучение на полном корпусе (~92 тыс. train-отзывов); аугментация для редких классов (замена синонимами, back-translation
  ru→en→ru, случайное удаление/перестановка слов); oversampling редкого класса или focal loss.
- *Модели:* другие предобученные модели для русского (`ai-forever/ruBert-base`, `cointegrated/rubert-tiny2` — быстрая,
  `DeepPavlov/rubert-base-cased-conversational`, `xlm-roberta-base`), постепенное размораживание слоёв (gradual unfreezing),
  дискриминативные learning rate по слоям, LSTM с предобученными fastText-эмбеддингами.
- *Обучение и оценка:* гиперпараметрический поиск (lr, batch size, число эпох, weight decay, dropout — Optuna),
  несколько seed'ов и кросс-валидация, подбор порогов/калибровка вероятностей, ансамблирование моделей.